# Transformer from scratch

## 1. The encoder block

Notebook 04 gave us every piece of one encoder layer. We stack them in the order the lecture uses (slide 69):

$$X \;\xrightarrow{\text{self-attention}}\; \operatorname{Add\,\&\,Norm} \;\xrightarrow{\text{feed-forward}}\; \operatorname{Add\,\&\,Norm}$$

- **Self-attention sublayer:** each English token looks at every other English token (notebook 03, section 2).
- **Add & Norm:** the residual connection and layer normalization around it (notebook 04, section 5).
- **Feed-forward sublayer:** each token is processed on its own (notebook 04, section 4).
- **Add & Norm:** again, around the feed-forward sublayer.
- The output has the same shape as the input, $T \times d_{\text{model}}$, so this block can be stacked $N$ times (the lecture uses $N = 6$).

#### 1.1 One encoder layer, step by step

![One token, bear, going through self-attention, Add and Norm, feed-forward, Add and Norm](images/encoder_layer_one_token.svg)

We follow "bear" through one encoder layer, reusing the numbers of notebook 04:

- $F_1(x)$: self-attention output (notebook 03, section 2 and notebook 04, section 5.3).
- $y_1 = x + F_1$, then $z_1 = \operatorname{LayerNorm}(y_1)$: the first Add & Norm (notebook 04, section 5).
- $F_2(z_1)$: feed-forward output, applied to $z_1$, not to $x$ (notebook 04, section 4).
- $y_2 = z_1 + F_2$, then $z_2 = \operatorname{LayerNorm}(y_2)$: the second Add & Norm.

$z_2$ is the output of this one layer. It has the same shape as $x$ ($d_{\text{model}} = 4$ numbers), so it becomes the input $x$ of the next encoder layer. With $N = 6$ layers, this repeats 6 times before reaching the decoder.

#### 1.2 All four tokens at once

![All four tokens going through one encoder layer: x, self-attention, Add and Norm, feed-forward, Add and Norm](images/encoder_layer_four_tokens.svg)

Same encoder layer as 1.1, now for "A cute teddy bear" together:

- Each row is one token. Every stage keeps shape $4 \times 4 = T \times d_{\text{model}}$.
- **Self-attention** needs all rows at once: each token's query looks at every key, so $F_1$ mixes information across the sentence.
- **Add & Norm** and the **feed-forward** act on each row alone (same weights for every token).
- The highlighted "bear" row matches section 1.1 exactly.
- $z_2$ is the output of this one layer. It becomes the input of the next encoder layer ($N = 6$ times in the lecture).

## 2. The decoder block

The decoder writes the French sentence one token at a time. One decoder layer has three sublayers (lecture slide 69), each wrapped in Add & Norm:

$$X_{\text{dec}} \;\xrightarrow{\text{masked self-attention}}\; \operatorname{Add\,\&\,Norm} \;\xrightarrow{\text{cross-attention}}\; \operatorname{Add\,\&\,Norm} \;\xrightarrow{\text{feed-forward}}\; \operatorname{Add\,\&\,Norm}$$

- **Masked self-attention:** each French token looks only at French tokens written so far (notebook 03, section 3). The mask hides the future.
- **Cross-attention:** each French token looks at the encoder output (the English vectors). Queries come from the decoder; keys and values come from the encoder (notebook 04, section 3).
- **Feed-forward:** same as in the encoder, one token at a time (notebook 04, section 4).
- **Add & Norm** around each of the three sublayers (notebook 04, section 5).
- Shape stays $T_{\text{dec}} \times d_{\text{model}}$, so these layers also stack $N = 6$ times.

#### 2.1 One decoder layer, step by step

![One French token, ours, going through masked self-attention, cross-attention, feed-forward, each with Add and Norm](images/decoder_layer_one_token.svg)

We follow "ours" through one decoder layer. French so far is "Un ours" (two rows). The English table is the encoder output $z_2$ from section 1.2.

- $F_1(x)$: **masked self-attention** among French tokens. "ours" may look at "Un" and itself, not at later words (notebook 03, section 3).
- $z_1 = \operatorname{LayerNorm}(x + F_1)$: first Add & Norm.
- $F_2(z_1)$: **cross-attention** into the encoder. Queries from the decoder; keys and values from English. "ours" puts the most weight on "bear" ($0.37$) (notebook 04, section 3).
- $z_2 = \operatorname{LayerNorm}(z_1 + F_2)$: second Add & Norm.
- $F_3(z_2)$: **feed-forward**, same network as the encoder (notebook 04, section 4).
- $z_3 = \operatorname{LayerNorm}(z_2 + F_3)$: third Add & Norm.

$z_3$ is the output of this one decoder layer ($d_{\text{model}} = 4$ numbers). It becomes the input of the next decoder layer ($N = 6$).

#### 2.2 Both French tokens at once

![Both French tokens Un and ours going through one decoder layer](images/decoder_layer_two_tokens.svg)

Same decoder layer as 2.1, now for "Un" and "ours" together:

- Each row is one French token. Every stage keeps shape $2 \times 4 = T_{\text{dec}} \times d_{\text{model}}$.
- **Masked self-attention:** "Un" sees only itself; "ours" may see "Un" and itself. The future stays hidden.
- **Cross-attention:** both rows look into the same 4 English encoder vectors (section 1.2).
- **Add & Norm** and the **feed-forward** act on each row alone.
- The highlighted "ours" row matches section 2.1 exactly.
- $z_3$ is the output of this one decoder layer. It becomes the input of the next one ($N = 6$).

## 3. End to end: the translation problem

Sections 1 and 2 built the encoder and decoder layers in detail. Here we put them back into one story: **what problem are we solving, and how does the architecture help?**

![End-to-end Transformer: English through the encoder, French through the decoder with cross-attention, softmax picks the next word](images/end_to_end_architecture.svg)

**The problem.** Translate English into French, one French word at a time:

> A cute teddy bear is reading.  →  Un ours en peluche mignon lit.

**How the architecture helps**

1. **Encode English once**
   - Embedding + position: words become vectors that know order.
   - Encoder layers (self-attention + feed-forward, section 1): each English word looks at every English word.
   - Result: **encoder memory** — one context-aware vector per English word.

2. **Decode French step by step**
   - Input so far: `[BOS] Un ours \ldots`
   - Embedding + position (same idea as the encoder, own tables).
   - **Masked self-attention:** only look at French tokens already written (section 2).
   - **Cross-attention:** look up English meanings in the encoder memory (e.g. "ours" → "bear").
   - Feed-forward: process each French position on its own.

3. **Pick the next French word**
   - Vocabulary size $= V$ (picture shows a toy subset).
   - $z$ has shape $1 \times d_{\text{model}}$ (here $1 \times 4$).
   - $W_{\text{out}}$ has shape $d_{\text{model}} \times V$ (here $4 \times V$).
   - $z W_{\text{out}}$ has shape $1 \times V$: **one logit (raw score) per vocab word**.
   - Softmax keeps shape $1 \times V$ (probabilities).
   - Pick the highest (here "en" at $0.73$), append it, run again until `[EOS]`.

**One-line picture.** Encoder understands English → decoder writes French while looking back → score all of $V$ → pick one.

## 4. Check in PyTorch

Two checks: our section-1 encoder numbers still match, then a built-in encoder–decoder forward pass so we can read the shapes end to end.

#### 4.1 Our encoder layer still matches section 1

In [1]:
import torch
import torch.nn.functional as F

# English token vectors from section 1 (A, cute, teddy, bear)
english = torch.tensor([[ 0.1, 0.0,  0.3, -0.1],
                        [ 0.2, 0.5, -0.1,  0.4],
                        [ 0.7, 0.1, -0.3,  0.8],
                        [-0.1, 0.9,  0.4,  0.2]])

# Same made-up weights as section 1 / notebook 04
W_Q = torch.tensor([[0., 0.], [1., -1.], [1., -1.], [1., 0.]])
W_K = torch.tensor([[0., -1.], [1., 0.], [-1., 1.], [1., 1.]])
W_V = torch.tensor([[0., 1., 1., 0.], [1., 0., 0., 1.], [1., 0., 1., 0.], [0., 0., 0., 1.]])
W1 = torch.tensor([[ 1.,  0., -1.,  1.,  0., -1.,  1.,  0.],
                   [ 0.,  1.,  1., -1.,  0.,  1., -1.,  1.],
                   [-1.,  1.,  0.,  0.,  1., -1.,  0., -1.],
                   [ 1., -1.,  1.,  0., -1.,  1.,  1.,  0.]])
W2 = torch.tensor([[1., 0., 0.,  1.],
                   [0., 1., 0.,  0.],
                   [0., 0., 1.,  0.],
                   [1., 0., 0., -1.],
                   [0., 1., 1.,  0.],
                   [0., 0., 1.,  1.],
                   [1., 1., 0.,  0.],
                   [0., 0., 1.,  0.]])


def layernorm(y, eps=1e-5):
    mu = y.mean(-1, keepdim=True)
    var = y.var(-1, unbiased=False, keepdim=True)
    return (y - mu) / torch.sqrt(var + eps)


def self_attention(X):
    # Each English token looks at every English token
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    return F.softmax(Q @ K.T / (Q.shape[-1] ** 0.5), dim=-1) @ V


# One encoder layer = attention + Add&Norm + feed-forward + Add&Norm
attn_out = self_attention(english)
after_attn = layernorm(english + attn_out)          # Add & Norm
ff_out = torch.relu(after_attn @ W1) @ W2           # feed-forward
encoder_out = layernorm(after_attn + ff_out)        # Add & Norm

print("english:    ", tuple(english.shape), "  T_enc x d_model")
print("encoder_out:", tuple(encoder_out.shape), "  same shape after one layer")
print("bear matches section 1.1:",
      torch.allclose(encoder_out[3], torch.tensor([-1.30, -0.43, 1.42, 0.32]), atol=5e-3))


english:     (4, 4)   T_enc x d_model
encoder_out: (4, 4)   same shape after one layer
bear matches section 1.1: True


- **`english`:** the English token vectors (section 1 input).
- **`self_attention`:** each English word looks at every English word.
- **`layernorm(english + attn_out)`:** Add & Norm after attention.
- **`ff_out`:** feed-forward on each token alone.
- **`encoder_out`:** output of one encoder layer — same shape as the input; "bear" matches section 1.1.

#### 4.2 Shape walk-through with built-in layers

Same English → French story as section 3, using PyTorch's `TransformerEncoderLayer` and `TransformerDecoderLayer`.

In [2]:
import torch

torch.manual_seed(0)

d_model, d_ff, nhead = 4, 8, 2
T_enc, T_dec = 4, 2
V = 8   # toy vocab size (real V is much larger)

# Built-in layers: same roles as section 3
encoder_layer = torch.nn.TransformerEncoderLayer(
    d_model=d_model, nhead=nhead, dim_feedforward=d_ff, dropout=0.0, batch_first=True
)
decoder_layer = torch.nn.TransformerDecoderLayer(
    d_model=d_model, nhead=nhead, dim_feedforward=d_ff, dropout=0.0, batch_first=True
)

# batch=1 for PyTorch: 1 x T x d_model
english = torch.randn(1, T_enc, d_model)            # English sentence in
french_so_far = torch.randn(1, T_dec, d_model)      # [BOS]-style French prefix

memory = encoder_layer(english)                     # encoder memory
causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(T_dec)
decoder_out = decoder_layer(french_so_far, memory, tgt_mask=causal_mask)

# Output head for the last French position (the one about to predict next)
W_out = torch.randn(d_model, V)                     # d_model x V
z = decoder_out[0, -1]                              # 1 x 4  (one position)
logits = z @ W_out                                  # 1 x V  (one score per vocab word)
probs = torch.softmax(logits, dim=-1)               # 1 x V  (probabilities)

print("english:       ", tuple(english.shape), "       1 x T_enc x d_model")
print("memory:        ", tuple(memory.shape), "       encoder output, same shape")
print("french_so_far: ", tuple(french_so_far.shape), "       1 x T_dec x d_model")
print("decoder_out:   ", tuple(decoder_out.shape), "       same as french_so_far")
print("z:             ", tuple(z.shape), "          last French position")
print("logits:        ", tuple(logits.shape), "           one logit per vocab word (V)")
print("probs:         ", tuple(probs.shape), "           probabilities over V")
print("probs sum ~= 1:", torch.allclose(probs.sum(), torch.tensor(1.0)))


english:        (1, 4, 4)        1 x T_enc x d_model
memory:         (1, 4, 4)        encoder output, same shape
french_so_far:  (1, 2, 4)        1 x T_dec x d_model
decoder_out:    (1, 2, 4)        same as french_so_far
z:              (4,)           last French position
logits:         (8,)            one logit per vocab word (V)
probs:          (8,)            probabilities over V
probs sum ~= 1: True


- **`english`:** English sentence in. Shape $1 \times T_{\text{encoder}} \times d_{\text{model}}$ (the leading $1$ is the batch size PyTorch wants).
- **`memory`:** encoder output — English meanings the decoder will look up. Same shape as `english`.
- **`french_so_far`:** French tokens written so far (like `[BOS] Un ours`). Shape $1 \times T_{\text{decoder}} \times d_{\text{model}}$.
- **`causal_mask`:** hides future French tokens (notebook 03, section 3).
- **`decoder_out`:** decoder layer output. Same shape as `french_so_far`.
- **`z`:** the last French position's vector (size $d_{\text{model}} = 4$) — the one used to pick the next word.
- **`W_out`:** $d_{\text{model}} \times V$ map to the vocabulary (random here).
- **`logits`:** one raw score per vocab word. Shape $V$.
- **`probs`:** softmax of the logits. Still shape $V$, and the numbers add up to 1.

## 5. Summary

Section 3's picture is the full map. Here is the same path as a checklist.

**Path (English → French)**

1. English tokens → embedding + position → **encoder** layers (self-attention + feed-forward + Add & Norm).
2. Result: **encoder memory** — one vector per English token.
3. French so far → embedding + position → **decoder** layers (masked self-attention + cross-attention into memory + feed-forward + Add & Norm).
4. Last decoder vector $z$ → $z W_{\text{out}}$ → logits → softmax → pick next token from vocabulary $V$.
5. Append that token, run the decoder again, stop at `[EOS]`.

**Shapes to remember**

$T$ = sequence length (number of tokens). $T_{\text{encoder}}$ = English length; $T_{\text{decoder}}$ = French length so far.

| Piece | Shape |
|---|---|
| Encoder activations | $T_{\text{encoder}} \times d_{\text{model}}$ (or $1 \times T_{\text{encoder}} \times d_{\text{model}}$ with a batch axis; the **1** = one example) |
| Decoder activations | $T_{\text{decoder}} \times d_{\text{model}}$ (or $1 \times T_{\text{decoder}} \times d_{\text{model}}$ with a batch axis; the **1** = one example) |
| Encoder memory | $T_{\text{encoder}} \times d_{\text{model}}$ |
| Cross-attention scores | $T_{\text{decoder}} \times T_{\text{encoder}}$ (one row per French token, one column per English token) |
| $z$ (one step) | $1 \times d_{\text{model}}$ |
| $W_{\text{out}}$ | $d_{\text{model}} \times V$ |
| Logits / probs | $1 \times V$ |

**Where the details live**

- Tokenization & embeddings: notebook 01
- Attention & causal mask: notebook 03
- Positional encoding, multi-head, cross-attention, feed-forward, Add & Norm: notebook 04
- Full encoder / decoder blocks & end-to-end path: this notebook